# Fase 1 — Caracterização estatística univariada

Este notebook caracteriza as distribuições marginais dos **12 canais ERA5** e do **alvo de radar** do dataset CorrDiff 2011–2024.

A análise mantém deliberadamente dois domínios do alvo separados:

- `target_stored_valid`: valor efetivamente armazenado no Zarr, após `log1p`;
- `target_radar_legend_valid`: `expm1(target)`, que reconstrói o valor numérico da legenda do radar **após** o `clip(min=0)` aplicado pelo builder.

> **Importante:** o builder chama esse campo de refletividade, mas não declara formalmente a unidade física. Portanto, neste notebook o eixo é chamado de **valor numérico da legenda de refletividade**, e não automaticamente de dBZ.

A Fase 1 usa uma amostra de blocos Zarr espalhados ao longo de todo o período para estudar forma de distribuição, quantis, assimetria e caudas. Quando os artefatos da Fase 0 estão disponíveis, médias/desvios/min/max exatos e taxas globais de eventos da varredura completa são usados como referência.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

cwd = Path.cwd()
if (cwd / "analysis_outputs").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "analysis_outputs").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

OUT = PROJECT_ROOT / "analysis_outputs" / "01_univariate"
PHASE0 = PROJECT_ROOT / "analysis_outputs" / "00_quality"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Phase 1 output:", OUT)
assert OUT.exists(), f"Execute primeiro scripts/01_compute_univariate_stats.py. Diretório ausente: {OUT}"


In [ ]:
def load_parquet(name):
    path = OUT / name
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()

def load_json_file(path):
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

summary = load_json_file(OUT / "analysis_summary.json")
input_summary = load_parquet("input_summary.parquet")
input_quantiles = load_parquet("input_quantiles.parquet")
input_hist = load_parquet("input_histograms.parquet")
rh_quality = load_parquet("relative_humidity_quality.parquet")
target_summary = load_parquet("target_summary.parquet")
target_quantiles = load_parquet("target_quantiles.parquet")
target_hist = load_parquet("target_histograms.parquet")
target_rates_sampled = load_parquet("target_threshold_rates_sampled.parquet")
target_rates_reference = load_parquet("target_threshold_rates_reference.parquet")
sampling_blocks = load_parquet("sampling_blocks.parquet")
samples = np.load(OUT / "univariate_samples.npz")

summary


## 1. Cobertura da amostragem usada na Fase 1

A Fase 0 já percorreu o dataset completo. Para não repetir desnecessariamente dezenas de bilhões de valores ERA5, a Fase 1 lê **blocos completos do Zarr** distribuídos ao longo de todo o dataset e compara as estatísticas amostrais com as estatísticas exatas da Fase 0.

O objetivo desta seção é verificar se a amostra é representativa antes de interpretar histogramas, quantis, assimetria e curtose.


In [ ]:
sampling_info = summary.get("sampling", {})
pd.DataFrame([
    {"Item": "Patches solicitados", "Valor": sampling_info.get("requested_sample_patches")},
    {"Item": "Patches efetivamente lidos", "Valor": sampling_info.get("sampled_patch_count")},
    {"Item": "Blocos selecionados", "Valor": sampling_info.get("selected_block_count")},
    {"Item": "Tamanho do bloco", "Valor": sampling_info.get("block_size")},
    {"Item": "Estratégia", "Valor": sampling_info.get("strategy")},
    {"Item": "Reservoir máximo por variável", "Valor": sampling_info.get("reservoir_values_per_channel_limit")},
])


In [ ]:
cols = [
    "channel", "expected_unit", "sampled_mean", "sampled_std",
    "phase0_exact_mean", "phase0_exact_std",
    "sample_mean_diff_in_phase0_std", "sample_std_relative_diff_vs_phase0"
]
available_cols = [c for c in cols if c in input_summary.columns]
input_summary[available_cols].round(6)


In [ ]:
if "sample_mean_diff_in_phase0_std" in input_summary.columns:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(input_summary["channel"], input_summary["sample_mean_diff_in_phase0_std"])
    ax.axhline(0.10, linestyle="--", linewidth=1, label="limiar diagnóstico = 0,10σ")
    ax.set_ylabel("|média amostral - média exata| / σ exato")
    ax.set_xlabel("Canal ERA5")
    ax.set_title("Representatividade da amostra da Fase 1")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.show()


## 2. Resumo univariado dos 12 canais ERA5

As unidades abaixo são **unidades convencionais esperadas para as variáveis ERA5**, usadas como indicação de interpretação. A análise numérica em si é feita sobre os valores presentes no Zarr.


In [ ]:
show_cols = [
    "channel_index", "channel", "expected_unit",
    "sampled_mean", "sampled_std", "sampled_min", "sampled_max",
    "sampled_skewness_from_reservoir", "sampled_excess_kurtosis_from_reservoir",
    "sampled_finite_ratio"
]
input_summary[[c for c in show_cols if c in input_summary.columns]].round(5)


### 2.1 Quantis

Os quantis são calculados sobre um conjunto limitado e distribuído de valores (`reservoir`) obtidos dos blocos amostrados. Eles são adequados para caracterizar a forma global das distribuições, mas não substituem uma climatologia espaço-temporal desduplicada.


In [ ]:
quantile_table = input_quantiles.pivot(index="name", columns="quantile", values="value")
ordered = [c for c in ["p001", "p01", "p05", "p10", "p25", "p50", "p75", "p90", "p95", "p99", "p999"] if c in quantile_table.columns]
quantile_table[ordered].round(4)


### 2.2 Histogramas dos canais ERA5

Os gráficos abaixo mostram a distribuição efetivamente vista pelo treinamento em patches. Como `stride=16` e `patch_size=32`, pixels centrais do campo completo aparecem em mais de um patch; portanto, estes histogramas descrevem o **dataset de treinamento**, não uma climatologia espacial sem sobreposição.


In [ ]:
channels = input_summary.sort_values("channel_index")["channel"].tolist()
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.ravel()

for ax, channel in zip(axes, channels):
    h = input_hist[input_hist["name"] == channel]
    if h.empty:
        ax.set_visible(False)
        continue
    width = h["bin_right"] - h["bin_left"]
    ax.bar(h["bin_center"], h["ratio"], width=width, align="center")
    unit = input_summary.loc[input_summary["channel"] == channel, "expected_unit"].iloc[0]
    ax.set_title(channel)
    ax.set_xlabel(unit)
    ax.set_ylabel("Fração")

plt.suptitle("Distribuições marginais dos 12 canais ERA5", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


## 3. Temperatura: visualização adicional em °C

O dataset preserva a temperatura em Kelvin. Para interpretação visual, podemos converter apenas a cópia usada no gráfico por `°C = K - 273,15`, sem modificar o Zarr.


In [ ]:
temp_channels = [c for c in ["t2m", "t_850", "t_500"] if f"input__{c}" in samples.files]
fig, axes = plt.subplots(1, len(temp_channels), figsize=(5 * max(len(temp_channels), 1), 4))
if len(temp_channels) == 1:
    axes = [axes]

for ax, channel in zip(axes, temp_channels):
    values_c = samples[f"input__{channel}"].astype(float) - 273.15
    ax.hist(values_c[np.isfinite(values_c)], bins=100, density=True)
    ax.set_title(channel)
    ax.set_xlabel("°C")
    ax.set_ylabel("Densidade")

plt.tight_layout()
plt.show()


## 4. Umidade relativa: valores fora de [0, 100]

Na Fase 0 observamos mínimos abaixo de 0% e máximos acima de 100% para `r_850` e `r_500`. Aqui quantificamos esse fenômeno na amostra da Fase 1, **sem recortar os dados**.

A decisão de limitar valores para modelagem só deve ocorrer depois de entendermos a origem e a frequência desses casos.


In [ ]:
rh_quality.round(8)


In [ ]:
if not rh_quality.empty:
    plot = rh_quality.set_index("channel")[["below_0_ratio", "above_100_ratio"]] * 100
    ax = plot.plot(kind="bar", figsize=(8, 4.5))
    ax.set_ylabel("Percentual dos valores amostrados (%)")
    ax.set_xlabel("Canal")
    ax.set_title("Umidade relativa fora do intervalo físico nominal [0,100]")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()


## 5. Assimetria e caudas

Assimetria (`skewness`) e curtose excedente são úteis para detectar variáveis com distribuições fortemente não gaussianas, caudas pesadas e possível necessidade de tratamento específico de normalização ou análise por regimes.


In [ ]:
shape_df = input_summary[[
    "channel", "sampled_skewness_from_reservoir", "sampled_excess_kurtosis_from_reservoir"
]].copy()
shape_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(shape_df["channel"], shape_df["sampled_skewness_from_reservoir"])
axes[0].axhline(0, linewidth=1)
axes[0].set_title("Assimetria")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(shape_df["channel"], shape_df["sampled_excess_kurtosis_from_reservoir"])
axes[1].axhline(0, linewidth=1)
axes[1].set_title("Curtose excedente")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 6. Radar: domínio armazenado versus domínio reconstruído

O builder grava:

\[
Y = \log(1 + \max(R,0))
\]

onde `R` é o valor numérico proveniente do cache do radar. Portanto:

\[
R_{clip} = \exp(Y) - 1
\]

O segundo domínio permite interpretar novamente os níveis numéricos da legenda. Valores negativos eventualmente presentes antes do `clip` **não podem ser recuperados**.


In [ ]:
target_summary.round(6)


In [ ]:
q_target = target_quantiles.pivot(index="name", columns="quantile", values="value")
ordered = [c for c in ["p001", "p01", "p05", "p10", "p25", "p50", "p75", "p90", "p95", "p99", "p999"] if c in q_target.columns]
q_target[ordered].round(5)


### 6.1 Massa em zero e cauda positiva

Como o radar é altamente esparso, um histograma único tende a esconder completamente a distribuição dos eventos positivos. Por isso visualizamos separadamente:

1. distribuição completa, dominada por zero;
2. distribuição apenas dos valores positivos.


In [ ]:
radar = samples["target_radar_legend_valid"].astype(float)
radar = radar[np.isfinite(radar)]
radar_pos = radar[radar > 0]

zero_ratio_sample = float(np.mean(radar == 0)) if radar.size else np.nan
positive_ratio_sample = float(np.mean(radar > 0)) if radar.size else np.nan

pd.DataFrame([
    {"Métrica": "Fração zero na amostra visual", "Valor": zero_ratio_sample},
    {"Métrica": "Fração positiva na amostra visual", "Valor": positive_ratio_sample},
    {"Métrica": "Valores válidos no reservoir", "Valor": radar.size},
    {"Métrica": "Valores positivos no reservoir", "Valor": radar_pos.size},
])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(radar, bins=120)
axes[0].set_yscale("log")
axes[0].set_xlabel("Valor numérico da legenda de refletividade")
axes[0].set_ylabel("Contagem (escala log)")
axes[0].set_title("Radar — todos os valores válidos")

if radar_pos.size:
    axes[1].hist(radar_pos, bins=120)
axes[1].set_xlabel("Valor numérico da legenda de refletividade")
axes[1].set_ylabel("Contagem")
axes[1].set_title("Radar — apenas valores positivos")

plt.tight_layout()
plt.show()


### 6.2 ECDF dos eventos positivos

A ECDF ajuda a responder, entre os pixels com eco positivo, qual fração está abaixo de cada intensidade numérica da legenda.


In [ ]:
if radar_pos.size:
    sorted_pos = np.sort(radar_pos)
    ecdf = np.arange(1, sorted_pos.size + 1) / sorted_pos.size
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(sorted_pos, ecdf)
    for level in [20, 25, 30, 35, 40, 45, 50]:
        ax.axvline(level, linewidth=0.8, linestyle="--")
    ax.set_xlabel("Valor numérico da legenda de refletividade")
    ax.set_ylabel("Fração acumulada")
    ax.set_title("ECDF — valores positivos do radar")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


## 7. Desbalanceamento por limiar

Quando disponível, usamos `target_threshold_rates_reference.parquet`, produzido pela Fase 0 a partir da **varredura completa do target**. Caso contrário, usamos a estimativa dos blocos amostrados da Fase 1.

Duas grandezas devem ser distinguidas:

- `event_pixel_ratio`: proporção de pixels válidos que atingem o limiar;
- `event_patch_ratio`: proporção de patches que possuem pelo menos um pixel atingindo o limiar.


In [ ]:
rates = target_rates_reference.copy() if not target_rates_reference.empty else target_rates_sampled.copy()
rates[[
    "condition", "legend_value_threshold", "event_pixel_ratio", "event_patch_ratio"
]].round(8)


In [ ]:
if not rates.empty:
    x = np.arange(len(rates))
    width = 0.38
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width/2, rates["event_pixel_ratio"] * 100, width, label="pixels válidos")
    ax.bar(x + width/2, rates["event_patch_ratio"] * 100, width, label="patches com evento")
    ax.set_xticks(x)
    ax.set_xticklabels(rates["condition"])
    ax.set_yscale("log")
    ax.set_ylabel("Percentual (%) — escala log")
    ax.set_xlabel("Limiar da legenda")
    ax.set_title("Desbalanceamento do alvo por intensidade")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 8. Comparação padronizada entre os canais

A padronização abaixo é apenas exploratória. Ela permite comparar a forma das distribuições em uma mesma escala, usando os valores amostrados. Não altera o dataset.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for channel in channels:
    key = f"input__{channel}"
    if key not in samples.files:
        continue
    values = samples[key].astype(float)
    values = values[np.isfinite(values)]
    if values.size == 0 or values.std() == 0:
        continue
    z = (values - values.mean()) / values.std()
    # histograma normalizado em grade comum para evitar milhões de pontos
    hist, edges = np.histogram(z, bins=np.linspace(-5, 5, 161), density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    ax.plot(centers, hist, linewidth=1, label=channel)

ax.set_xlim(-5, 5)
ax.set_xlabel("z-score na amostra")
ax.set_ylabel("Densidade")
ax.set_title("Forma padronizada das distribuições ERA5")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()


## 9. Síntese automática da Fase 1

Esta célula transforma os principais diagnósticos em uma tabela compacta para registro metodológico.


In [ ]:
rows = []
rep = summary.get("sampling", {}).get("representativeness_vs_phase0", {})
rows.append({
    "Checagem": "Representatividade da amostra vs Fase 0",
    "Status": "OK" if rep.get("max_abs_sample_mean_difference_in_phase0_std", 0) <= 0.10 else "REVISAR",
    "Detalhe": rep,
})

for _, row in rh_quality.iterrows():
    rows.append({
        "Checagem": f"{row['channel']} fora de [0,100]",
        "Status": "OBSERVAR" if row["outside_0_100_ratio"] > 0 else "OK",
        "Detalhe": f"{row['outside_0_100_ratio']:.6%}",
    })

if not rates.empty:
    gt0 = rates[rates["threshold"] == "gt_0"]
    if not gt0.empty:
        r = gt0.iloc[0]
        rows.append({
            "Checagem": "Esparsidade do radar",
            "Status": "IMPORTANTE",
            "Detalhe": f"pixels >0: {r['event_pixel_ratio']:.6%}; patches com >0: {r['event_patch_ratio']:.6%}",
        })

rows.append({
    "Checagem": "Unidade física da legenda do radar",
    "Status": "REVISAR",
    "Detalhe": "não declarada pelo builder; não rotular como dBZ sem documentação da fonte",
})

pd.DataFrame(rows)


## 10. Critérios para encerrar a Fase 1

A Fase 1 pode ser considerada concluída quando:

- a amostra distribuída reproduz adequadamente médias/desvios exatos da Fase 0;
- todas as 12 distribuições ERA5 foram caracterizadas por média, desvio, extremos, quantis, assimetria e curtose;
- valores de umidade relativa fora de `[0,100]` foram quantificados e documentados, sem tratamento silencioso;
- a distribuição do radar foi analisada separando `target` armazenado e `expm1(target)`;
- a massa em zero e as taxas por limiar foram quantificadas;
- ficou documentado que essas distribuições são **ponderadas pela estrutura de patches sobrepostos**;
- a unidade física da legenda do radar continua explicitamente separada da transformação numérica do builder.

Com isso, a próxima etapa é a **Fase 2 — variáveis derivadas fisicamente**, incluindo magnitude do vento, diferenças verticais e cisalhamento entre níveis, antes de avançarmos para as relações conjuntas ERA5 × radar.
